# Attention From Scratch

- Module: 10 Transformers and LLM Foundations
- Topic: scaled dot-product attention and masking
- Estimated runtime: 10 minutes
- Prerequisites: linear algebra, softmax, matrix multiplication, NumPy basics
- Expected outputs: attention weights, context vectors, scaling comparison plot, causal mask demo


## Learning goals

By the end of this notebook you should be able to:

- build queries, keys, and values as explicit matrix multiplications;
- compute self-attention with row-wise softmax;
- verify that each attention row sums to one;
- see numerically why the `1 / sqrt(d_k)` factor matters; and
- apply a causal mask that blocks future positions.


In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

SEED = 7
rng = np.random.default_rng(SEED)
np.set_printoptions(precision=3, suppress=True)


def softmax_rows(scores: np.ndarray) -> np.ndarray:
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


SEED


## A toy sequence and its projections

We use a sequence length of `T = 4` and model width `d_model = 4`.
The projection matrices map token embeddings into query, key, and value spaces.


In [ ]:
X = np.array(
    [
        [1.0, 0.0, 1.0, 0.5],
        [0.0, 1.0, 0.5, 1.0],
        [1.0, 1.0, 0.0, 0.5],
        [0.5, 0.0, 1.0, 1.0],
    ]
)

W_Q = np.array(
    [
        [0.8, -0.2],
        [0.1, 0.7],
        [0.6, 0.3],
        [-0.4, 0.5],
    ]
)
W_K = np.array(
    [
        [0.5, 0.4],
        [-0.3, 0.8],
        [0.7, -0.1],
        [0.2, 0.6],
    ]
)
W_V = np.array(
    [
        [0.2, 1.0, -0.3],
        [0.9, -0.4, 0.2],
        [0.5, 0.3, 0.8],
        [-0.6, 0.7, 0.4],
    ]
)

d_k = W_Q.shape[1]
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("X shape:", X.shape)
print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)
Q


## Compute scaled dot-product attention

The score matrix is `Q @ K.T`.
We divide by `sqrt(d_k)`, apply a row-wise softmax, and then multiply by `V`.


In [ ]:
raw_scores = Q @ K.T
scaled_scores = raw_scores / np.sqrt(d_k)
attention_weights = softmax_rows(scaled_scores)
context = attention_weights @ V

print("Raw score matrix:
", raw_scores)
print("
Scaled score matrix:
", scaled_scores)
print("
Attention weights:
", attention_weights)
print("
Row sums:", attention_weights.sum(axis=1))
print("
Context vectors:
", context)


## Why the scaling factor matters

If the key dimension grows, unscaled dot products become larger in magnitude.
The next cell measures the average maximum logit magnitude and the average row entropy with and without scaling.


In [ ]:
def entropy_rows(weights: np.ndarray) -> np.ndarray:
    safe = np.clip(weights, 1e-12, 1.0)
    return -(safe * np.log(safe)).sum(axis=1)


def sample_attention_stats(d_k: int, trials: int = 200) -> tuple[float, float, float, float]:
    unscaled_max = []
    scaled_max = []
    unscaled_entropy = []
    scaled_entropy = []
    for _ in range(trials):
        q = rng.normal(size=(4, d_k))
        k = rng.normal(size=(4, d_k))
        logits = q @ k.T
        logits_scaled = logits / np.sqrt(d_k)
        unscaled_weights = softmax_rows(logits)
        scaled_weights = softmax_rows(logits_scaled)
        unscaled_max.append(np.abs(logits).max())
        scaled_max.append(np.abs(logits_scaled).max())
        unscaled_entropy.append(entropy_rows(unscaled_weights).mean())
        scaled_entropy.append(entropy_rows(scaled_weights).mean())
    return (
        float(np.mean(unscaled_max)),
        float(np.mean(scaled_max)),
        float(np.mean(unscaled_entropy)),
        float(np.mean(scaled_entropy)),
    )


dims = np.array([2, 4, 8, 16, 32, 64])
unscaled_maxima = []
scaled_maxima = []
unscaled_entropies = []
scaled_entropies = []

for dim in dims:
    max_unscaled, max_scaled, ent_unscaled, ent_scaled = sample_attention_stats(int(dim))
    unscaled_maxima.append(max_unscaled)
    scaled_maxima.append(max_scaled)
    unscaled_entropies.append(ent_unscaled)
    scaled_entropies.append(ent_scaled)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(dims, unscaled_maxima, marker="o", label="unscaled")
axes[0].plot(dims, scaled_maxima, marker="o", label="scaled")
axes[0].set_title("Average max |logit|")
axes[0].set_xlabel("d_k")
axes[0].set_ylabel("magnitude")
axes[0].legend()
axes[0].grid(alpha=0.3, linestyle="--")

axes[1].plot(dims, unscaled_entropies, marker="o", label="unscaled")
axes[1].plot(dims, scaled_entropies, marker="o", label="scaled")
axes[1].set_title("Average attention entropy")
axes[1].set_xlabel("d_k")
axes[1].set_ylabel("entropy")
axes[1].legend()
axes[1].grid(alpha=0.3, linestyle="--")

fig.tight_layout()
plt.show()


## Causal masking

A decoder should not attend to future positions.
We create an upper-triangular mask with `-inf` above the main diagonal before the softmax.


In [ ]:
mask = np.triu(np.full((X.shape[0], X.shape[0]), -np.inf), k=1)
masked_scores = scaled_scores + mask
masked_weights = softmax_rows(masked_scores)
masked_context = masked_weights @ V

print("Mask:
", mask)
print("
Masked attention weights:
", masked_weights)
print("
Upper-triangular entries are zero:")
print(np.triu(masked_weights, k=1))
print("
Masked context vectors:
", masked_context)


## Positional encoding reminder

Attention over `X` alone is permutation-equivariant.
A positional encoding breaks that symmetry by adding order-dependent vectors to the token embeddings.


In [ ]:
def sinusoidal_encoding(length: int, width: int) -> np.ndarray:
    positions = np.arange(length)[:, None]
    div_terms = np.exp(-np.log(10000.0) * np.arange(0, width, 2) / width)
    pe = np.zeros((length, width))
    pe[:, 0::2] = np.sin(positions * div_terms)
    pe[:, 1::2] = np.cos(positions * div_terms)
    return pe


PE = sinusoidal_encoding(length=X.shape[0], width=X.shape[1])
X_with_position = X + PE

print("Sinusoidal positional encoding:
", PE)
print("
First token embedding before/after adding position:")
print(X[0])
print(X_with_position[0])


## Summary

We built self-attention directly from matrices, confirmed that the weights form row-wise probability distributions, showed why `1 / sqrt(d_k)` keeps logits in a healthier range, and used a causal mask to recover decoder-style attention.

### Next steps

- Re-derive the same computation using your own dimensions and random seeds.
- Replace the toy projections with learned projections from a training loop.
- Compare single-head and multi-head attention on the same sequence.
